# 通用SPI模式与时序调试（数字波形模型）
默认只生成软件波形，不驱动GPIO/SPI、不下载Overlay。修改参数后可检查分频、边沿、CS及位序。模型通过不代表芯片命令或真实电气时序正确。


In [ ]:
from pathlib import Path
import sys, json, csv
ROOT = Path.cwd()
if not (ROOT/'scripts/spi_contract.py').is_file(): ROOT = ROOT.parent
assert (ROOT/'scripts/spi_contract.py').is_file(), 'Open notebook in the skill directory or notebooks subdirectory'
sys.path.insert(0, str(ROOT/'scripts'))
from spi_contract import SpiConfig, divider, build_trace, validate_trace


## 1. 配置：模式、帧长、位序、时钟与CS时序
时间单位为PL时钟周期。MODE=0..3，BIT_ORDER=msb/lsb。以下值仅为通用离线例子，非特定ADC/DAC手册参数。


In [ ]:
CLOCK_HZ = 50_000_000
REQUESTED_SCK_HZ = 1_000_000
MODE = 0
FRAME_BITS = 8
BIT_ORDER = 'msb'
CS_SETUP_CYCLES = 2
CS_HOLD_CYCLES = 2
DATA_SETUP_CYCLES = 1
DATA_HOLD_CYCLES = 1
TX_WORD = 0xA6
EXPECTED_RX_WORD = 0x53
config = SpiConfig(clock_hz=CLOCK_HZ, sck_hz=REQUESTED_SCK_HZ, mode=MODE, frame_bits=FRAME_BITS, bit_order=BIT_ORDER, cs_setup_cycles=CS_SETUP_CYCLES, cs_hold_cycles=CS_HOLD_CYCLES, data_setup_cycles=DATA_SETUP_CYCLES, data_hold_cycles=DATA_HOLD_CYCLES)
print('实现速率/分频：', divider(config))
print('CPOL/CPHA：', config.cpol, config.cpha)


## 2. 合成波形或输入数字采样CSV
CSV列必须为t_cycles,cs,sck,mosi,miso，时间严格递增，逻辑值0/1。只支持单次CS低事务；采样时基由你明确提供。


In [ ]:
TRACE_CSV = None  # 实际数字trace文件；None生成合成trace
if TRACE_CSV is None:
    trace = build_trace(config, TX_WORD, EXPECTED_RX_WORD)
    source = 'SYNTHETIC_TRACE'
else:
    with Path(TRACE_CSV).open(newline='', encoding='utf-8') as f:
        trace = [{k:int(v) for k,v in row.items()} for row in csv.DictReader(f)]
    source = 'USER_TRACE_NOT_AUTHENTICATED'
result = validate_trace(trace, config, TX_WORD, EXPECTED_RX_WORD)
result['source'] = source
print(json.dumps(result, indent=2))
assert result['passed'], result['quality_flags']


## 3. 可选保存
此处只保存报告，不发送任何信号。逻辑分析仪文件也不等同于完整模拟电气验证；需另查幅度、振铃、电平转换及芯片手册。


In [ ]:
SAVE_REPORT = False
if SAVE_REPORT:
    (ROOT/'examples/spi_debug_report.json').write_text(json.dumps(result, indent=2)+'\n', encoding='utf-8')
print('No hardware operations performed.')
